# Sweep Mode A — Pre-build sweep

A *sweep* runs many cases in one call and collects the results. openTEPES has three sweep modes; this notebook covers the first.

**Mode A (pre-build)** reads each case from its own input folder and builds and solves it independently. Use it when your cases are genuinely different folders, such as separate scenarios with many changed inputs.

When cases share most of their data, two faster modes avoid re-reading and rebuilding:
- [Mode B](5.2-Sweep-Mode-B-InMemory.ipynb) reads one baseline once and perturbs it.
- [Mode C](5.3-Sweep-Mode-C-Resolve.ipynb) builds the model once and re-solves it.

## The question

All three sweep notebooks answer the same question: **how does the total system cost change if electricity demand is higher?**

For Mode A we build two case folders: a base case and a high-demand case (demand scaled up by 10%).

In [1]:
import os, shutil
import pandas as pd

def coarse_copy(src, dst):
    """Copy a case folder and set a coarse time resolution so the example runs fast."""
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    p = os.path.join(dst, "oT_Data_Parameter_9n.csv")
    df = pd.read_csv(p)
    df.loc[:, "TimeStep"] = 24   # coarse resolution, just to keep this tutorial quick
    df.to_csv(p, index=False)

# Base scenario
coarse_copy("9n", "sweepA_base/9n")

# High-demand scenario: same case, electricity demand scaled up by 10%
coarse_copy("9n", "sweepA_high/9n")
dem = "sweepA_high/9n/oT_Data_Demand_9n.csv"
df = pd.read_csv(dem)
node_cols = [c for c in df.columns if c.startswith("Node_")]
df[node_cols] = df[node_cols] * 1.10
df.to_csv(dem, index=False)
print("Two scenario folders are ready.")

Two scenario folders are ready.


## Run the sweep

A `Case` describes one entry of the sweep: the folder that holds it (`dir_name`), the case name (`case_name`), where to write its results (`out_path`), and a short `label` for the summary.

`openTEPES_Runner.run` builds and solves each case and returns one summary row per case, in the order you listed them.

In [2]:
from openTEPES import openTEPES_Runner, openTEPES_Cases

cases = [
    openTEPES_Cases.Case("sweepA_base", "9n", out_path="sweepA_base/out", label="base"),
    openTEPES_Cases.Case("sweepA_high", "9n", out_path="sweepA_high/out", label="high_demand"),
]

records = openTEPES_Runner.run(
    cases, "appsi_highs",
    mode="pre-build", backend="serial",
    pIndOutputResults=0, pIndLogConsole=0,
)

Input data                             ****
Reading the CSV files                  ...  0 s
Reading    input data                  ...  0 s


Setting up input data                  ...  1 s
Setting up variables                   ...  0 s
Total cost o.f.      model formulation ****
Investment elec      model formulation ****
Period 2030, Scenario sc01, Stage st1
Generation oper o.f. model formulation ****


Investment & operation var constraints ****
Inertia, oper resr, demand constraints ****
Storage   scheduling       constraints ****
Unit commitment            constraints ****


Ramp and min up/down time  constraints ****
Network    switching model constraints ****
Network    operation model constraints ****
Problem solving                        #### 1


Termination condition:  optimal
  Total system                 cost [MEUR]  159.2111765517769  Constraints 41136  Variables 50966  Seconds 4
***** Period: 2030, Scenario: sc01, Stage: st1 ******
  Total generation  investment cost [MEUR]  0
  Total generation  retirement cost [MEUR]  0
  Total reservoir   investment cost [MEUR]  0.0
  Total network     investment cost [MEUR]  4.041225328215885
  Total H2   pipe   investment cost [MEUR]  0.0
  Total heat pipe   investment cost [MEUR]  0.0
  Total generation  operation  cost [MEUR]  155.16550622161094
  Total consumption operation  cost [MEUR]  0.0028532975065870803
  Total emission               cost [MEUR]  0.0
  Total network losses penalty cost [MEUR]  0.0015917044432770529
  Total reliability electr     cost [MEUR]  0.0
Writing            investment results  ...  0 s
Writing          cost summary results  ...  0 s
Writing           KPI summary results  ...  0 s


...

Ramp and min up/down time  constraints ****
Network    switching model constraints ****
Network    operation model constraints ****


Problem solving                        #### 1


Termination condition:  optimal
  Total system                 cost [MEUR]  199.2279074895432  Constraints 41136  Variables 50966  Seconds 3
***** Period: 2030, Scenario: sc01, Stage: st1 ******
  Total generation  investment cost [MEUR]  0
  Total generation  retirement cost [MEUR]  0
  Total reservoir   investment cost [MEUR]  0.0
  Total network     investment cost [MEUR]  5.337109032001712
  Total H2   pipe   investment cost [MEUR]  0.0
  Total heat pipe   investment cost [MEUR]  0.0
  Total generation  operation  cost [MEUR]  193.88620417509878
  Total consumption operation  cost [MEUR]  0.0028444742272815686
  Total emission               cost [MEUR]  0.0
  Total network losses penalty cost [MEUR]  0.001749808215068684
  Total reliability electr     cost [MEUR]  0.0
Writing            investment results  ...  0 s
Writing          cost summary results  ...  0 s
Writing           KPI summary results  ...  0 s


Writing elect network summary results  ...  0 s


Writing              economic results  ...  0 s


In [3]:
pd.DataFrame(records)[["label", "status", "total_cost_meur"]]

,label,status,total_cost_meur
0,base,optimal,159.211177
1,high_demand,optimal,199.227907


## What just happened

Each case was solved on its own, and the total system cost rises in the high-demand scenario. A case that fails gets `status="error"` instead of stopping the whole sweep.

**Run in parallel.** Switch the backend to use several CPU cores:

```python
records = openTEPES_Runner.run(cases, "appsi_highs", mode="pre-build",
                               backend="multiprocessing", n_workers=2)
```

`backend="joblib"` also works (needs `pip install joblib`).

**Merge the results.** Pass `aggregate_to="sweepA_merged"` to stack every case's result tables into one long table per result type, tagged by case label.

When your cases share most of their inputs, Mode A re-reads the same data for each one. The next notebook, [Mode B](5.2-Sweep-Mode-B-InMemory.ipynb), avoids that.